Atlas's CMS is a file-backed content system implemented in Python. The `at cms` Typer commands call `Store` in `src/atlas/cms_store.py` to read and mutate records. `src/atlas/cms_site.py` projects saved content into a Quarto frontend. The source of a post remains an ordinary `.ipynb`; generated site files are a separate projection.

This article examines the data model, `Store`, revision and recovery mechanisms, validation, and the sync path. It is the first article in [Building Atlas](../collections/building-atlas.qmd); the next covers the [notebook CLI](building-atlas-the-notebook-cli.qmd).

## Architecture and file ownership

The CMS has no database server. Its state is a set of YAML records, authored files, and managed images inside a workspace:

```text
backend/cms.yaml                   # Schema marker
backend/posts/<id>.yaml            # Post record
backend/collections/<id>.yaml      # Collection record and ordered members
backend/resume/<id>.yaml           # Résumé section and position
content/posts/<id>.ipynb           # Post cells, outputs, attachments, options
content/collections/<id>.yaml      # Authored collection fields
content/resume/<id>.yaml           # Authored résumé entry fields
content/resume.yaml                # Résumé title and summary
content/images/                    # Imported image bytes
frontend/_cms/                     # Editable Quarto templates
frontend/posts/, collections/      # Generated notebook and page copies
.atlas/cms/                        # Lock, journal, staging, recovery files
```

The backend records and authored files must be versioned together. Frontend projections and `.atlas/cms/` runtime files are ignored. Templates and Quarto configuration remain editable; sync replaces generated copies.

The CLI constructs a `Store` for the selected root in `src/atlas/cms.py`:

```python
@app.callback()
def configure(ctx: typer.Context, root: Path = ROOT_PATH, as_json: bool = False):
    ctx.obj = {"store": Store(root), "json": as_json}
```

This excerpt omits the CLI annotations, but preserves the relationship: commands get the same root-scoped store from the Typer context. `--root` selects another initialized workspace; without it, Atlas uses the checkout root. `Store.require_initialized()` checks `backend/cms.yaml` for schema `3` and validates the résumé profile before record access. `at cms init` initializes a workspace.

## The CMS Store and its records

`Store` is the persistence boundary. Its methods resolve workspace paths, load and validate records, create and update content, manage images, and commit file changes. The CMS and notebook command groups both construct a `Store`, so notebook saves use the same locking and record-update machinery.

Post records include the stable ID, title, status, short description, timestamps, tags, thumbnail path, notebook source path and digest, optional code path, related post IDs, and private notes and to-dos. The notebook itself is not embedded in that YAML. Collections and résumé entries have a split model: `Store.get()` reads the backend record, loads its authored YAML source, merges the expected content fields, and validates the resulting in-memory record.

For example, a collection's title, short description, long description, and optional cover caption belong in `content/collections/<id>.yaml`; membership, status, and cover path belong in its backend record. A résumé entry's title, subtitle, and body are authored fields; section and position are backend fields. `record_bytes()` omits authored fields when serializing the backend record, and `content_bytes()` serializes those fields separately. `get()` rejects authored fields found in the backend YAML and rejects unexpected fields in the content YAML.

The YAML loader also rejects duplicate or non-string mapping keys. Record validation checks exact fields and types, timezone-aware timestamps, valid IDs, source extensions, normalized tags, and ordered post references without duplicates or self-references. `references()` separately loads referenced posts to ensure they exist. Private notes and to-dos remain in the record and are excluded from public site metadata; changing only those fields does not update the public modification date.

### The record read path in code

Here is the central part of `Store.get()` from `src/atlas/cms_store.py` (the file-existence and identity checks preceding it are omitted):

```python
kind = identity.split("/")[0]
authored = CONTENT_FIELDS.get(kind, ())
if any(key in record for key in authored):
    raise ValueError(
        f"Editable fields for {identity} belong in its content/ source."
    )
revision_raw = raw
if authored:
    source_raw = self.path(record.get("source_path")).read_bytes()
    content = CONTENT_DEFAULTS.get(kind, {}) | decode(source_raw)
    if set(content) != set(authored):
        raise ValueError(
            f"Expected only {', '.join(authored)} in {record['source_path']}."
        )
    record.update(content)
    revision_raw += b"\0" + source_raw
self.validate(record)
return {"record": record, "revision": digest(revision_raw)}
```

`raw` is the backend file's bytes. For a collection or résumé entry, `CONTENT_FIELDS` identifies fields loaded from its authored source. The zero byte separates the two byte sequences before hashing, and validation sees the merged record. For a post, the notebook is not merged into the record revision: notebook operations check its digest separately.

### How a mutation reaches disk

The beginning of `Store.update()` establishes the write boundary (the excerpt stops before image handling and final commit):

```python
with self.lock():
    loaded = self.get(identity)
    self.check_revision(loaded, expected)
    old = loaded["record"]
    kind = identity.split("/")[0]
    forbidden = {
        "id",
        "created_at",
        "updated_at",
        "source_digest",
        "source_path",
    }
    if set(patch) & forbidden or not set(patch) <= set(old):
        raise ValueError("Unknown or system-managed field in patch.")
    record = old | patch
    if "tags" in patch:
        record["tags"] = normalized_tags(patch["tags"])
```

The record is reloaded **inside** the lock before comparing the expected revision. That makes the check and subsequent update one cooperating writer operation. System-managed fields cannot be patched directly; later code validates the candidate record, updates authored content when necessary, and sends the resulting file changes to `commit()`.

The writer lock itself also starts recovery before letting a new operation proceed:

```python
@contextmanager
def lock(self):
    lock_path = self.path(".atlas/cms/writer.lock")
    lock_path.parent.mkdir(parents=True, exist_ok=True)
    with lock_path.open("a+b") as handle:
        fcntl.flock(handle, fcntl.LOCK_EX)
        try:
            self.recover()
            yield
        finally:
            fcntl.flock(handle, fcntl.LOCK_UN)
```

`yield` is where the caller's operation runs. `finally` releases the advisory lock even when that operation fails. Recovery can itself reject a conflicting pending save before the caller changes anything.


## Identity, paths, and creation

IDs have the form `post/<slug>`, `collection/<slug>`, or `resume/<slug>`. `record_path()` validates the ID before mapping it to a backend YAML file. `Store.new()` derives an initial slug from the title and appends an eight-character UUID suffix if that record ID already exists. The ID and source filename are not changed when a title is edited.

Paths stored in records are workspace-relative. `Store.path()` rejects absolute paths, `..` components, backslashes, and symlinks along the resolved path. This limits the store's file operations to supported paths inside the workspace. New content cannot overwrite an existing source file.

For a post, `Store.new()` creates an empty notebook and a draft record in the same logical save. It records the SHA-256 digest of the notebook bytes as `source_digest`. For collections and résumé entries, it prepares an authored YAML source and its digest instead. The command returns the record and its revision; use the returned ID rather than predicting whether a collision occurred.

```sh
uv run at cms post new "Understanding Attention" \
  --description "How attention works" --tags "python,transformers"
uv run at cms --json post show post/understanding-attention
```

The second command assumes creation returned that exact ID. The `--json` option is useful for lists; `show` includes both the record and the revision token.

## Revisions and stale writes

The store uses SHA-256 for both source digests and record revisions, but they represent different things. For posts, `Store.get()` computes the record revision from the backend YAML bytes. For collections and résumé entries, it hashes the backend bytes, a zero byte separator, and the authored YAML bytes. A direct edit to a collection introduction therefore changes its next loaded revision even before sync updates its stored `source_digest`.

A CMS mutation can supply `--expected-revision`. `Store.update()` acquires the writer lock, reloads the record, and compares the supplied token with the current revision before applying the patch. On mismatch it raises `Conflict`; the CLI reports a conflict with exit code `3`. The caller should reload, reconcile, and retry with the new token.

```sh
uv run at cms post show post/understanding-attention
uv run at cms post edit post/understanding-attention \
  --description "An introduction to attention" \
  --expected-revision '<revision-from-show>'
```

Notebook edits need an additional check because a post's record revision alone does not include its notebook bytes. `Store.write_notebook()` checks the record revision and the digest of the notebook bytes read before editing. Notebook CLI revision tokens are prefixed `notebook:` and are obtained from `cat --with-revision` or `outline`; they are not interchangeable with CMS record revisions. On a successful notebook change, the store commits the notebook and its updated source digest and modification date together.

Revision checks are opt-in for CMS updates when the caller omits `--expected-revision`; they are not a substitute for coordinating simultaneous external editor saves. Recovery files preserve failed editor input and execution results where applicable.

## Locking and recoverable saves

The store serializes cooperating writers with an exclusive `fcntl.flock` on `.atlas/cms/writer.lock`. On entering the lock, it first calls `recover()` to finish any pending save. The lock covers store operations; an external editor that ignores it can still change a file, which is why digest and revision checks matter.

A logical operation can touch several files: creating a post writes both a notebook and its record; an update may write authored YAML, a record, and an imported image. `Store.commit()` builds a journal in `.atlas/cms/pending.json`. Each entry contains the workspace-relative path and the before and after bytes encoded as base64; `None` represents a missing file or deletion. Unchanged files are omitted.

The installation sequence is:

1. Write the journal with `_install()`, which writes a temporary file in the destination directory, flushes and `fsync`s it, replaces the destination, and `fsync`s the directory.
2. Call `recover()`. Before applying anything, it checks that every target currently matches either its journaled before or after state.
3. Install the after state for each entry, or unlink a deleted target, then remove the journal.
4. If a target matches neither state, raise `Conflict` and leave the journal for repair rather than overwriting an unrecognized change.

This is a recoverable multi-file operation, not a claim that all files become visible in one instantaneous filesystem transaction. Individual replacements are atomic; the journal lets a later cooperating operation roll an interrupted save forward. Recovery can refuse to proceed if an intervening edit conflicts. `Store.preserve()` separately saves recovery copies such as failed editor input.

## Validation, references, and publication

Structural validity and publication readiness are separate checks. Drafts can be incomplete, but their records must still satisfy the store's schema. `Store.ready()` adds publication requirements:

| Kind | Required for publication |
| --- | --- |
| Post | Nonempty description, verified thumbnail, and a notebook with at least one cell containing source text. |
| Collection | Nonempty short and long descriptions, verified cover, and at least one published member. |

A record's title is required by structural validation. Publishing is an explicit status transition; publishing a collection does not publish its posts. Subsequent edits can make a published record unready, so public sync checks readiness again. `at cms check` reports invalid references, duplicate post source ownership, and readiness problems for drafts as well as published records. An unfinished draft can therefore cause it to exit with status `1`.

Each notebook source path can belong to only one post, including drafts. Both sync modes check duplicate ownership before updating source digests or generated files. Removing a post also refuses a source claimed by another post. Collection membership is an ordered list of exact post IDs; the CLI checks positions and uses the loaded record revision for the final update.

Images pass through `Store.image()`: it reads the file, rejects inputs over 20 MiB, verifies PNG, JPEG, WebP, or GIF data with Pillow, and returns a managed path based on the SHA-256 digest of the original bytes. The bytes are copied, not re-encoded. Notebook figure imports also support SVG through the notebook operation. The source file remains in place.

## Sync: from saved sources to Quarto

`generate(store, preview)` in `src/atlas/cms_site.py` implements the projection. Under the store lock, it loads posts, collections, and résumé entries; rejects duplicate notebook ownership; reads source files; and compares each source's SHA-256 digest with its record. A changed external source updates `source_digest` and `updated_at` at sync time. The timestamp reflects observation by sync, not the filesystem modification time. The authored source is not rewritten.

Preview selects both drafts and published records. Public sync selects only published posts and collections, validates their references, and checks readiness. Résumé entries are included in both modes. The generator then stages output under `.atlas/cms/` before installing the generated files through `Store.commit()`.

For each selected post, sync copies the parsed notebook in memory, inserts a generated metadata cell into the frontend copy, and writes `frontend/posts/<id>.ipynb`. It converts `.qmd` links in Markdown to `.html` in the projection, projects managed images, and copies eligible notebook-relative sidecars. The source notebook is left intact. The collection projection writes a `.qmd` page with the authored introduction and a generated table derived from ordered member IDs; only selected posts appear in that table. Listing and résumé pages are generated from templates and saved content.

The staged projection replaces generated frontend files and removes stale generated files, while retaining `_metadata.yml`, templates, and configuration. Sync also creates a résumé PDF input under `.atlas/cms/`; sync itself does not render the PDF.

```sh
uv run at cms sync --preview  # Include drafts in generated copies
uv run at cms sync            # Published content only
make preview                  # Sync, refresh PDF if needed, serve preview
make build                    # Sync, refresh PDF if needed, render public site
```

The preview and build entrypoints call sync before Quarto and pass `--no-execute`. They render saved notebook outputs; they do not run code cells. The PDF renderer uses an input/output hash cache so unchanged inputs and output can be reused. Publication controls build eligibility, while deployment is a separate operation.

## Testing the persistence boundary

The CMS tests create a `Store` rooted in a temporary workspace and exercise the CLI and store without changing the real content. Relevant cases in `tests/test_cms.py` cover stale revision rejection, duplicate source ownership in both sync modes, deterministic sync without source mutation, removal of stale generated files, and recovery after an interrupted notebook-and-record save. These tests target the boundaries that are easy to lose when several files represent one logical edit.

The [notebook CLI article](building-atlas-the-notebook-cli.qmd) follows the other half of the system: locating cells, editing them with revision tokens, importing figures, and executing code explicitly.



